# ST446 WT2026 - Assigment 1

**Task information:** In this assignment, you will analyse global COVID-19 pandemic data. You will progressively move from low-level distributed processing (Spark RDDs) to structured analytics (Spark DataFrames and Spark SQL) and finally to NoSQL analytics using Bigtable.

The assignment consists of **4 parts**, as follows:
* **Part A:** Spark RDD (4 questions)
* **Part B:** Spark DataFrames (3 questions)
* **Part C:** Spark SQL (2 questions)
* **Part D:** Bigtable (2 questions)

Your **overall goal** is not only to compute results, but to justify analytical choices, understand trade-offs between APIs, and reflect on how AI tools (if used) supported or influenced your reasoning.

<hr>

## Setup

In [2]:
# Python libraries
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Spark libraries
from pyspark.sql import SparkSession
# Spark SQL data types
from pyspark.sql.types import *
# Spark SQL functions
import pyspark.sql.functions as sql_f

<hr>

## Data

Check the data source/provider and information about the dataset, especially metadata.

* Dataset: **COVID-19 Pandemic**
* Data source: Our World in Data
* Latest version (including metadata/column information): https://docs.owid.io/projects/covid/en/latest/dataset.html
* Project information: https://ourworldindata.org/coronavirus


In [3]:
# -----------------------------------------------
# Dataset download and Hadoop ingestion
# -----------------------------------------------
!wget https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
!hadoop fs -put -f owid-covid-data.csv /

--2026-03-03 19:17:24--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘owid-covid-data.csv’

owid-covid-data.csv 100%[===================>]  93.83M   472MB/s    in 0.2s    

2026-03-03 19:17:24 (472 MB/s) - ‘owid-covid-data.csv’ saved [98391483/98391483]



In [4]:
# -----------------------------------------------
# Check Hadoop masternode
# -----------------------------------------------
!hadoop fs -ls /
# confirms the csv is successfuly in distributed storage

Found 4 items
-rw-r--r--   2 root hadoop   98391483 2026-03-03 19:17 /owid-covid-data.csv
drwxrwxrwt   - hdfs hadoop          0 2026-03-03 19:14 /tmp
drwxrwxrwt   - hdfs hadoop          0 2026-03-03 19:15 /user
drwxrwxrwt   - hdfs hadoop          0 2026-03-03 19:13 /var


In [5]:
# -----------------------------------------------
# HDFS file path
# Adjust to match your cluster name and file path
# -----------------------------------------------
filename = "hdfs://st446-cluster-w04-m:8020/owid-covid-data.csv"

<hr>

## Part A — Spark RDDs (4 questions)

**Goal:** Understand distributed data processing, key/value transformations, aggregation, and ranking logic.

---

### Dataset loading & exploration

As usual in most analytical scenarios, you can **load the raw data into a RDD** and keep it unchanged throughout your analytical pipeline. For each specific task, this RDD can be loaded into a new RDD and subject to the required transformations/actions.

In [6]:
# Raw RDD
raw_rdd = spark.sparkContext.textFile(filename)
# Check header (column names)
header = raw_rdd.first()
columns = header.split(",")
# Print out column names
for i, col_name in enumerate(columns):
    print(f"Index: {i}: {col_name}")

Index: 0: iso_code
Index: 1: continent
Index: 2: location
Index: 3: date
Index: 4: total_cases
Index: 5: new_cases
Index: 6: new_cases_smoothed
Index: 7: total_deaths
Index: 8: new_deaths
Index: 9: new_deaths_smoothed
Index: 10: total_cases_per_million
Index: 11: new_cases_per_million
Index: 12: new_cases_smoothed_per_million
Index: 13: total_deaths_per_million
Index: 14: new_deaths_per_million
Index: 15: new_deaths_smoothed_per_million
Index: 16: reproduction_rate
Index: 17: icu_patients
Index: 18: icu_patients_per_million
Index: 19: hosp_patients
Index: 20: hosp_patients_per_million
Index: 21: weekly_icu_admissions
Index: 22: weekly_icu_admissions_per_million
Index: 23: weekly_hosp_admissions
Index: 24: weekly_hosp_admissions_per_million
Index: 25: total_tests
Index: 26: new_tests
Index: 27: total_tests_per_thousand
Index: 28: new_tests_per_thousand
Index: 29: new_tests_smoothed
Index: 30: new_tests_smoothed_per_thousand
Index: 31: positive_rate
Index: 32: tests_per_case
Index: 33: t

---

### Question 1 - Baseline RDD

Using the **raw RDD** loaded in the previous cell:
* create a reduced **baseline RDD** named ` covid_rdd` to answer all questions in Part A
* show three samples from the baseline RDD

The baseline RDD needs only a few columns:

```
# | Index    | Column             |
# | -------- | ------------------ |
# | 0        | iso_code           |
# | 1        | continent          |
# | 2        | location           |
# | 3        | date               |
# | 4        | total_cases        |
# | 5        | new_cases          |
# | 17       | icu_patients       |
# | 19       | hosp_patients      |
# | 34       | total_vaccinations |
# | 38       | new_vaccinations   |
# | 62       | population         |

```

**IMPORTANT:** OWID frequently adds new columns to the dataset. **You must verify column positions using the dataset metadata before hard-coding indices. Your code should not assume indices blindly without validating the schema**.

**OWID Dataset — baseline RDD assumptions:**
* `iso_code` refers to three-letter country codes (e.g. USA, FRA, DEU) or to prefix `OWID_` for non-country level aggregated data (e.g. continents like 'Europe' etc)
* Rows where `continent` is empty (or `None`) can also correspond to non-country aggregates (e.g. European Union, Internatioanl, income groups)
* `total_cases` and `total_vaccinations` are cumulative, so `max(total_cases)` gives final count
* Dates are in `YYYY-MM-DD` format (lexicographically sortable)

**Requirements and Constraints:**
* Verify column positions in the raw RDD before hard-coding indices
* Baseline RDD must be named `covid_rdd`
* Remove headers when generating the baseline RDD
* Map only the necessary columns from the raw RDD into `covid_rdd`

**Expected output:** the baseline RDD should have the following structure:

```
# | Index    | Column             |
# | -------- | ------------------ |
# | 0        | iso_code           |
# | 1        | continent          |
# | 2        | location           |
# | 3        | date               |
# | 4        | total_cases        |
# | 5        | new_cases          |
# | 6        | icu_patients       |
# | 7        | hosp_patients      |
# | 8        | total_vaccinations |
# | 9        | new_vaccinations   |
# | 10       | population         |
```

A sample should look like:
```
[('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-05',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772')
```

In [7]:
## CODE

# verifying columns exist
col_to_position = {name: i for i, name in enumerate(columns)}
needed_names = ["iso_code", "continent", "location", "date", "total_cases", "new_cases", "icu_patients", "hosp_patients", "total_vaccinations", "new_vaccinations", "population"]

missing = []
for name in needed_names: 
    if name not in col_to_position: 
        missing.append(name)
        
if missing:
    raise KeyError(f"Missing columns: {', '.join(missing)}")
    
    # testing it by adding fake colname to needed names
    # it works!


In [8]:
print(col_to_position)

{'iso_code': 0, 'continent': 1, 'location': 2, 'date': 3, 'total_cases': 4, 'new_cases': 5, 'new_cases_smoothed': 6, 'total_deaths': 7, 'new_deaths': 8, 'new_deaths_smoothed': 9, 'total_cases_per_million': 10, 'new_cases_per_million': 11, 'new_cases_smoothed_per_million': 12, 'total_deaths_per_million': 13, 'new_deaths_per_million': 14, 'new_deaths_smoothed_per_million': 15, 'reproduction_rate': 16, 'icu_patients': 17, 'icu_patients_per_million': 18, 'hosp_patients': 19, 'hosp_patients_per_million': 20, 'weekly_icu_admissions': 21, 'weekly_icu_admissions_per_million': 22, 'weekly_hosp_admissions': 23, 'weekly_hosp_admissions_per_million': 24, 'total_tests': 25, 'new_tests': 26, 'total_tests_per_thousand': 27, 'new_tests_per_thousand': 28, 'new_tests_smoothed': 29, 'new_tests_smoothed_per_thousand': 30, 'positive_rate': 31, 'tests_per_case': 32, 'tests_units': 33, 'total_vaccinations': 34, 'people_vaccinated': 35, 'people_fully_vaccinated': 36, 'total_boosters': 37, 'new_vaccinations': 

In [9]:
# creating baseline RDD called covid_rdd

index_ls = [col_to_position[name] for name in needed_names]
data_rdd = raw_rdd.filter(lambda x: x!= header)
parsed_rdd = data_rdd.map(lambda x: x.split(","))

covid_rdd = parsed_rdd.map(lambda fields: tuple(fields[i] for i in index_ls))

# printing out 3 samples
covid_rdd.take(3)

[('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-05',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772'),
 ('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-06',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772'),
 ('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-07',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772')]

---

### Question 2: Hierarchical Aggregation (Continent -> Country)

Using the **baseline RDD**:
* Compute the total number of confirmed COVID-19 cases per continent level
* For each continent, identify the top 2 countries with the highest cumulative number of confirmed cases
* Show results from both RDDs

**Requirements and Constraints:**
* Use RDD transformations and actions only (no DataFrames or SQL)
* Filter out rows that correspond to aggregates and rows with missing continent
* Use the maximum cumulative total cases per country before aggregating at continent level

**Expected output:**
* One RDD with total cases per continent: `[(continent, total cases), ...]`
* One RDD with top 2 countries per continent by total cases, descending order: `[(continent, [(country1, total cases), (country2, total cases)]), ...]`
* Show results from both RDDs



In [10]:
# helper function (strings -> int, blanks -> 0)
to_int = lambda x: int(x) if x not in ("", None) else 0

# 1) keep only country-level rows (drop OWID_* and missing continent)
countries = covid_rdd.filter(lambda r: r[1] not in ("", None) and not r[0].startswith("OWID_"))

# 2) max cumulative total_cases per country (iso_code)
#    value holds (continent, location, max_total_cases)
per_country_max = (
    countries
    .map(lambda r: (r[0], (r[1], r[2], to_int(r[4]))))                 # (iso, (continent, country, total_cases))
    .reduceByKey(lambda a, b: a if a[2] >= b[2] else b)                # keep max total_cases
)

# RDD A: total cases per continent
continent_totals = (
    per_country_max
    .map(lambda kv: (kv[1][0], kv[1][2]))                              # (continent, cases)
    .reduceByKey(lambda a, b: a + b)
)

# RDD B: top 2 countries per continent by max total_cases (desc)
top2_by_continent = (
    per_country_max
    .map(lambda kv: (kv[1][0], (kv[1][1], kv[1][2])))                  # (continent, (country, cases))
    .groupByKey()
    .mapValues(lambda it: sorted(it, key=lambda x: x[1], reverse=True)[:2])
)

# show results from both RDDs
print(continent_totals.collect())
print(top2_by_continent.collect())

[('Asia', 301532347), ('Europe', 252642589), ('Africa', 13145540), ('South America', 68809418), ('Oceania', 15003352), ('North America', 124492666)]
[('Asia', [('China', 99373219), ('India', 45041748)]), ('Europe', [('France', 38997490), ('Germany', 38437756)]), ('Africa', [('South Africa', 4072765), ('Morocco', 1279115)]), ('South America', [('Brazil', 37511921), ('Argentina', 10101218)]), ('Oceania', [('Australia', 11861161), ('New Zealand', 2639048)]), ('North America', [('United States', 103436829), ('Mexico', 7619458)])]


<hr>

### Question 3 – Temporal Growth Analysis

Using the **baseline RDD**:
* Restrict the dataset to the period March–June 2021
* Compute weekly total new COVID-19 cases per country
* For each continent, identify the single country with the highest week-over-week growth in new cases during this period
* Show the resulting RDD

**Requirements and Constraints:**
* Use RDD transformations and actions only (no DataFrames or SQL)
* Filter out missing or null values from the required columns
* Define week using ISO week number derived from the date. Week-over-week growth is defined as: `growth = (current_week_total − previous_week_total)`
* Growth should be computed within each country only

**Expected output:** RDD with `[(continent, (country, highest week-over-week growth)), ...]`

In [11]:
from datetime import datetime

def safe_iso_week(date_str):
    try:
        if date_str in ("", None):
            return None
        return datetime.strptime(date_str, "%Y-%m-%d").isocalendar()[1]
    except Exception:
        return None

def to_int_safe(x):
    try:
        if x in ("", None):
            return 0
        return int(float(x)) 
    except Exception:
        return 0

def valid_row(r):
    # required cols: iso_code(0), continent(1), location(2), date(3), new_cases(5)
    try:
        return (
            r is not None and len(r) > 5
            and r[0] not in ("", None) and not r[0].startswith("OWID_")   # drop aggregates
            and r[1] not in ("", None)                                    # drop missing continent
            and r[2] not in ("", None)                                    # country name present
            and r[3] not in ("", None) and len(r[3]) >= 10                # date present
            and r[5] not in ("", None)                                    # new_cases present (can be "0")
            and r[3][4] == "-" and r[3][7] == "-"                         # quick YYYY-MM-DD check
        )
    except Exception:
        return False


In [12]:

# restricting to date march-june 2021
restricted = (
    covid_rdd
    .filter(valid_row)
    .filter(lambda r: "2021-03-01" <= r[3] <= "2021-06-30")
)

# computing total weekly new cases aggregated by country.
# key: (iso, week)  value: (continent, country, weekly_new_cases_sum)
weekly_country = (
    restricted
    .map(lambda r: ((r[0], safe_iso_week(r[3])), (r[1], r[2], to_int_safe(r[5]))))
    .filter(lambda kv: kv[0][1] is not None)  # drop rows with unparseable week
    .reduceByKey(lambda a, b: (a[0], a[1], a[2] + b[2]))
)

# computing weekly growth by country.
def compute_growth(records):
    # records: iterable of (week, continent, country, weekly_total)
    recs = sorted(records, key=lambda x: x[0])
    best = None
    prev_total = None
    for wk, cont, country, total in recs:
        if prev_total is not None:
            g = total - prev_total
            if best is None or g > best[2]:
                best = (cont, country, g)
        prev_total = total
    return best  # (continent, country, max_growth) or None

country_max_growth = (
    weekly_country
    .map(lambda kv: (kv[0][0], (kv[0][1], kv[1][0], kv[1][1], kv[1][2]))) 
    .groupByKey()
    .mapValues(compute_growth)
    .filter(lambda kv: kv[1] is not None)
    .map(lambda kv: (kv[1][0], (kv[1][1], kv[1][2])))  
)

# selecting country (in each continent) with highest week on week growth. 
result_rdd = country_max_growth.reduceByKey(lambda a, b: a if a[1] >= b[1] else b)

# show the resulting RDD
result_rdd.collect()

[('Asia', ('India', 742759)),
 ('Europe', ('France', 40845)),
 ('Africa', ('South Africa', 32958)),
 ('South America', ('Brazil', 80556)),
 ('Oceania', ('Papua New Guinea', 782)),
 ('North America', ('United States', 38267))]

<hr>

### Question 4: Vaccination Intensity Ranking

Using the **baseline RDD**:
* Compute an RDD with the average daily number of new COVID-19 vaccinations per country.
* Return the top 10 countries by average daily vaccinations.

**Requirements & Constraints:**
* Use RDD transformations and actions only
* Use valid (not null/missing) daily new vaccinations
* Do not use cumulative vaccination columns
* Include only country-level records (exclude continents, world totals, and income groups)
* Include only countries with at least 50 days of valid reported vaccination data
* Round numeric outputs to two decimal places

**Expected output:** RDD with `[(country, average daily vaccinations), ...]`

In [13]:

def to_float_safe(x):
    try:
        if x in ("", None):
            return None
        return float(x)
    except Exception:
        return None

In [14]:
ISO, CONT, LOC, DATE, NEW_VAX = 0, 1, 2, 3, 9

valid_daily = (
    covid_rdd
    .filter(lambda r: r[CONT] not in ("", None) and not r[ISO].startswith("OWID_"))
    .map(lambda r: (r[LOC], to_float_safe(r[NEW_VAX])))
    .filter(lambda kv: kv[1] is not None and kv[1] >= 0)
)

sum_count = (
    valid_daily
    .map(lambda kv: (kv[0], (kv[1], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
)

avg_daily = (
    sum_count
    .filter(lambda kv: kv[1][1] >= 50)
    .mapValues(lambda sc: round(sc[0] / sc[1], 2))
)

top10 = avg_daily.takeOrdered(10, key=lambda kv: -kv[1])

top10_rdd = spark.sparkContext.parallelize(top10)
top10_rdd.collect()

[('China', 5147424.47),
 ('India', 1731265.63),
 ('Indonesia', 808177.92),
 ('United States', 771588.55),
 ('Brazil', 724200.64),
 ('Pakistan', 587713.4),
 ('Japan', 489532.5),
 ('Mexico', 444550.72),
 ('Bangladesh', 441370.03),
 ('Philippines', 426459.62)]

<hr>

## Part B - Spark DataFrames (2 questions)

**Goal:** Understand structured analytics using transformations, aggregation, and ranking logic.

---

### Question 5: Data loading & baseline DataFrame

When loading data into Spark DataFrames, we usually define a mapping schema, based on `StructType` and `StructField`. This requires us to know the structure of the dataset and ensure that `StructField` names match the column names in the dataset; otherwise, NULL values are loaded. There are also other potential problems causing **column misalignment**, such as wrong separators and date formats.

For this part of the assignment, you must **create a baseline DataFrame** named `covid_df`:
* load the original dataset into a raw DataFrame (e.g., `raw_df`)
* filter out only the relevant columns (see table below) into a baseline DataFrame (e.g., `covid_df`)
* show three samples

**Requirements and Constraints:**
* instead of defining a schema, ask Spark to infer the schema for you. This is safer and avoids any potential column misalignment arising from the data
* note that we need **two additional columns** compared to the baseline RDD used in Part A.
* before attempting any DataFrames questions, print out some sample rows and check the inferred schema

**Expected output:** the baseline DataFrame should have this structure:

```
| Column name          | Description             |
| -------------------- | ----------------------- |
| `iso_code`           | Country codes           |
| `continent`          | Continent name          |
| `location`           | Country name            |
| `date`               | Observation date        |
| `total_cases`        | Cumulative COVID cases  |
| `new_cases`          | New cases               |
| `icu_patients`       | ICU patients            |
| `hosp_patients`      | Hospitalized patients   |
| `total_vaccinations` | Cumulative vaccinations |
| `new_vaccinations`   | Daily vaccinations      |
| `population`         | Total population        |
| `total_deaths`       | Total deaths            |
| `new_deaths`         | New deaths              |
| -------------------- | ----------------------- |
```

A sample will look like:

```
----------+----------+------------+----------+
|iso_code|continent|location |date      |total_cases|new_cases|icu_patients|hosp_patients|total_vaccinations|new_vaccinations|population|total_deaths|new_deaths|
+--------+---------+---------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|AUS     |Oceania  |Australia|2021-03-22|29196      |0        |1           |66           |312502            |30542           |26177410  |925         |0         |
```


In [15]:
from pyspark.sql import functions as F

raw_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv(filename)
)

raw_df.printSchema()
raw_df.show(3, truncate=False)

cols_needed = [
    "iso_code","continent","location","date",
    "total_cases","new_cases","icu_patients","hosp_patients",
    "total_vaccinations","new_vaccinations","population",
    "total_deaths","new_deaths",
]

covid_df = raw_df.select(*cols_needed)
covid_df.show(3, truncate=False)

26/03/03 19:17:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

+--------+---------+-----------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|iso_code|continent|location   |date      |total_cases|new_cases|icu_patients|hosp_patients|total_vaccinations|new_vaccinations|population|total_deaths|new_deaths|
+--------+---------+-----------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|AFG     |Asia     |Afghanistan|2020-01-05|0          |0        |NULL        |NULL         |NULL              |NULL            |41128772  |0           |0         |
|AFG     |Asia     |Afghanistan|2020-01-06|0          |0        |NULL        |NULL         |NULL              |NULL            |41128772  |0           |0         |
|AFG     |Asia     |Afghanistan|2020-01-07|0          |0        |NULL        |NULL         |NULL              |NULL            |41128772  |0           |0         |
+--------+------

In [16]:
covid_df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- total_vaccinations: long (nullable = true)
 |-- new_vaccinations: integer (nullable = true)
 |-- population: long (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)



In [17]:
# Null counts for key columns
key_cols = ["continent","location","date","population","new_cases","icu_patients","hosp_patients","new_vaccinations","total_vaccinations"]
nulls_df = covid_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"nulls_{c}")
    for c in key_cols
])
nulls_df.show(truncate=False)

# Quick numeric sanity (min/max) to catch separator/date issues
covid_df.select(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.min("population").alias("min_pop"),
    F.max("population").alias("max_pop"),
    F.max("total_cases").alias("max_total_cases"),
    F.max("total_vaccinations").alias("max_total_vaccinations")
).show(truncate=False)


+---------------+--------------+----------+----------------+---------------+------------------+-------------------+----------------------+------------------------+
|nulls_continent|nulls_location|nulls_date|nulls_population|nulls_new_cases|nulls_icu_patients|nulls_hosp_patients|nulls_new_vaccinations|nulls_total_vaccinations|
+---------------+--------------+----------+----------------+---------------+------------------+-------------------+----------------------+------------------------+
|26525          |0             |0         |0               |19276          |390319            |388779             |358464                |344018                  |
+---------------+--------------+----------+----------------+---------------+------------------+-------------------+----------------------+------------------------+

+----------+----------+-------+----------+---------------+----------------------+
|min_date  |max_date  |min_pop|max_pop   |max_total_cases|max_total_vaccinations|
+----------+---

---

### Question 6: System Pressure Indicator (ICU aggregations)

Using the **baseline DataFrame**:
* Compute a DataFrame with the average ICU patient load per country
* Show the top 10 countries

**Requirements and Constraints:**
* Use only country-level rows (exclude all continent / world aggregates)
* Include only countries with at least 50 non-null ICU observations
* Order results in descending order

**Expected output:** DataFrame with
* `location`: country name
* `icu_obs_count`: count of valid ICU observations
* `avg_icu_patients`: computed average ICU patient load

In [18]:
icu_df = (
    covid_df
    .filter(F.col("continent").isNotNull() & ~F.col("iso_code").startswith("OWID_"))
    .groupBy("location")
    .agg(
        F.count("icu_patients").alias("icu_obs_count"),
        F.avg("icu_patients").alias("avg_icu_patients"),
        F.round(F.expr("percentile_approx(icu_patients, 0.95)"), 2).alias("p95_icu_patients")
    )
    .filter(F.col("icu_obs_count") >= 50)
    .orderBy(F.desc("avg_icu_patients"))
    .withColumn("avg_icu_patients", F.round("avg_icu_patients", 2))
)

icu_df.show(10, truncate=False)

+--------------+-------------+----------------+----------------+
|location      |icu_obs_count|avg_icu_patients|p95_icu_patients|
+--------------+-------------+----------------+----------------+
|United States |1381         |7703.35         |25328           |
|Argentina     |647          |2934.58         |6802            |
|France        |1109         |2046.52         |5254            |
|Germany       |1194         |1771.24         |4844            |
|Spain         |1045         |1060.6          |3021            |
|United Kingdom|781          |900.5           |3164            |
|South Africa  |873          |803.65          |2368            |
|Chile         |1248         |795.66          |3067            |
|Japan         |203          |706.84          |1916            |
|Italy         |1627         |695.05          |3260            |
+--------------+-------------+----------------+----------------+
only showing top 10 rows



In [19]:
# making sure that I show the WHOLE dataframe, as per the README file. 
icu_df.toPandas()

,location,icu_obs_count,avg_icu_patients,p95_icu_patients
0,United States,1381,7703.35,25328
1,Argentina,647,2934.58,6802
2,France,1109,2046.52,5254
3,Germany,1194,1771.24,4844
4,Spain,1045,1060.60,3021
5,United Kingdom,781,900.50,3164
6,South Africa,873,803.65,2368
7,Chile,1248,795.66,3067
8,Japan,203,706.84,1916
9,Italy,1627,695.05,3260


---

### Question 7: Vaccination Momentum vs Population Size

Using the **baseline DataFrame**, for each country, compute:
* total number of vaccinations administered
* average daily vaccinations
* number of days with reported vaccination data
* `vaccinations_per_1000`, given by `(total vaccinations/population) * 1000`

**Requirements:**
* Restrict the dataset to country-level observations only (exclude continents and other aggregates)
* Consider only rows with valid `new_vaccinations` and `population` data
* Include only countries with at least 50 reporting days
* Round numeric outputs to two decimal places
* Order the results by`vaccinations_per_1000` in descending order

**Expected output:** DataFrame with:
* `location`
* `population`
* `total_vaccination`
* `avg_daily_vaccinations`
* `reporting_days`
* `vaccinations_per_1000`



In [23]:
vaccination_df = (
    covid_df
    .filter(F.col("continent").isNotNull() & ~F.col("iso_code").startswith("OWID_"))
    .filter(F.col("new_vaccinations").isNotNull() & F.col("population").isNotNull())
    .withColumn("new_vaccinations", F.col("new_vaccinations").cast("double"))
    .groupBy("location", "population")
    .agg(
        F.sum("new_vaccinations").alias("total_vaccination"),
        F.avg("new_vaccinations").alias("avg_daily_vaccinations"),
        F.count("new_vaccinations").alias("reporting_days"),
    )
    .filter(F.col("reporting_days") >= 50)
    .withColumn("vaccinations_per_1000", (F.col("total_vaccination") / F.col("population")) * 1000)
    .select(
        "location",
        "population",
        F.round("total_vaccination", 2).alias("total_vaccination"),
        F.round("avg_daily_vaccinations", 2).alias("avg_daily_vaccinations"),
        "reporting_days",
        F.round("vaccinations_per_1000", 2).alias("vaccinations_per_1000"),
    )
    .orderBy(F.desc("vaccinations_per_1000"))
)

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

vaccination_df.toPandas()

,location,population,total_vaccination,avg_daily_vaccinations,reporting_days,vaccinations_per_1000
0,Cuba,11212198,3.840438e+07,59634.13,644,3425.23
1,Chile,19603736,6.267192e+07,83562.57,750,3196.94
2,Japan,123951696,3.833039e+08,489532.50,783,3092.37
3,Gibraltar,32677,9.223400e+04,501.27,184,2822.60
4,Hong Kong,7488863,2.083967e+07,23336.70,893,2782.76
5,Belgium,11655923,3.145835e+07,29074.26,1082,2698.92
6,Peru,34049588,9.140872e+07,88403.02,1034,2684.58
7,Canada,38454328,1.022088e+08,82227.55,1243,2657.93
8,Uruguay,3422796,8.966322e+06,13815.60,649,2619.59
9,Singapore,5637022,1.472700e+07,22179.22,664,2612.55


<hr>

## Part C: Spark SQL (2 questions)

**Goal:** Use high-level, intuitive SQL-like statements over DataFrame structured data to produce country-level summary indicators.


---

### Question 8: Pandemic Severity Summary

Using the **baseline DataFrame** as a Spark SQL table, create a **country-level summary table** (`severity_df`) that supports cross-country risk comparison. For each country, compute:
* the maximum cumulative number of confirmed COVID-19 cases
* the maximum cumulative number of confirmed COVID-19 deaths
* `case_fatality_ratio` (CFR), defined as `(maximum total deaths / maximum total cases) * 100`

**Requirements:**
* Restrict the analysis to country-level observations only
* Include only rows with valid `total_cases` and `total_deaths` data
* CFR is approximated using maximum cumulative values; this does not account for temporal lag between cases and deaths
* Round numeric outputs to two decimal places
* Order the table by `case_fatality_ratio` in descending order

**Expected output:** `severity_df` DataFrame with:
* `location`
* `max_total_cases`
* `max_total_deaths`
* `case_fatality_ratio`


In [21]:
from pyspark.sql import functions as F

covid_df.createOrReplaceTempView("covid")

severity_df = spark.sql("""
SELECT
  location,
  ROUND(MAX(total_cases), 2)  AS max_total_cases,
  ROUND(MAX(total_deaths), 2) AS max_total_deaths,
  ROUND((MAX(total_deaths) / MAX(total_cases)) * 100, 2) AS case_fatality_ratio
FROM covid
WHERE
  continent IS NOT NULL
  AND iso_code NOT LIKE 'OWID_%'
  AND total_cases IS NOT NULL
  AND total_deaths IS NOT NULL
  AND total_cases > 0
GROUP BY location
ORDER BY case_fatality_ratio DESC
""")

severity_df.show(truncate=False)

+----------------------+---------------+----------------+-------------------+
|location              |max_total_cases|max_total_deaths|case_fatality_ratio|
+----------------------+---------------+----------------+-------------------+
|Yemen                 |11945          |2159            |18.07              |
|Sudan                 |63993          |5046            |7.89               |
|Syria                 |57423          |3163            |5.51               |
|Somalia               |27334          |1361            |4.98               |
|Peru                  |4526977        |220975          |4.88               |
|Egypt                 |516023         |24830           |4.81               |
|Mexico                |7619458        |334551          |4.39               |
|Bosnia and Herzegovina|403666         |16392           |4.06               |
|Liberia               |8090           |294             |3.63               |
|Afghanistan           |235214         |7998            |3.4    

In [18]:
# showing all rows and columns
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

severity_df.toPandas()

,location,max_total_cases,max_total_deaths,case_fatality_ratio
0,Yemen,11945,2159,18.07
1,Sudan,63993,5046,7.89
2,Syria,57423,3163,5.51
3,Somalia,27334,1361,4.98
4,Peru,4526977,220975,4.88
5,Egypt,516023,24830,4.81
6,Mexico,7619458,334551,4.39
7,Bosnia and Herzegovina,403666,16392,4.06
8,Liberia,8090,294,3.63
9,Afghanistan,235214,7998,3.40


---

### Question 9: Health System & Vaccination Readiness

Using the **baseline DataFrame** as a Spark SQL table, create a **country-level summary table** (`health_vax_df`) combining indicators of healthcare system burden and vaccination progress. Such summary table would enable comparison of countries' readiness to manage COVID-19.

For each country, compute:
* the average number of hospitalised COVID-19 patients over time
* the average number of ICU COVID-19 patients over time
* vaccinations per 1,000 people (`vax_per_1k`), defined as `(maximum total vaccinations / population) * 1000`

**Requirements and Constraints:**
* Restrict the analysis to country-level observations only
* For each metric, use non-null observations; however, only include countries where all required columns are non-null in the final result
* Round all numeric outputs to two decimal places
* Order the results by `vax_per_1k` in descending order
* Remember that `total_vaccinations` is a cumulative variable

**Expected output:** `health_vax_df` DataFrame with:
* `location`
* `avg_hosp_patients`
* `avg_icu_patients`
* `vax_per_1k`


In [19]:
covid_df.createOrReplaceTempView("covid")

health_vax_df = spark.sql("""
WITH base AS (
  SELECT
    location,
    hosp_patients,
    icu_patients,
    total_vaccinations,
    population
  FROM covid
  WHERE
    continent IS NOT NULL
    AND iso_code NOT LIKE 'OWID_%'
),
agg AS (
  SELECT
    location,
    ROUND(AVG(hosp_patients), 2) AS avg_hosp_patients,
    ROUND(AVG(icu_patients), 2)  AS avg_icu_patients,
    MAX(total_vaccinations)      AS max_total_vaccinations,
    MAX(population)              AS population
  FROM base
  GROUP BY location
)
SELECT
  location,
  avg_hosp_patients,
  avg_icu_patients,
  ROUND((max_total_vaccinations / population) * 1000, 2) AS vax_per_1k
FROM agg
WHERE
  avg_hosp_patients IS NOT NULL
  AND avg_icu_patients IS NOT NULL
  AND max_total_vaccinations IS NOT NULL
  AND population IS NOT NULL
  AND population > 0
ORDER BY vax_per_1k DESC
""")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

health_vax_df.toPandas()

,location,avg_hosp_patients,avg_icu_patients,vax_per_1k
0,Japan,12171.04,706.84,3095.95
1,Portugal,1232.34,184.79,2756.81
2,Belgium,1698.76,286.75,2699.86
3,Sweden,807.32,111.34,2683.84
4,Canada,3576.44,300.21,2675.31
5,Australia,1370.03,72.37,2662.09
6,Denmark,351.60,28.03,2543.50
7,South Korea,3510.05,307.32,2502.09
8,Italy,8286.24,695.05,2449.44
9,Finland,379.49,25.86,2405.29


<hr>

## PART D — Google Bigtable

**Goal:** Model a NoSQL (key/value pairs, in this case) table and run analytical queries.

Using the summary tables (`severity_df` and `health_vax_df`) that you computed in Part C, you will:
* Create a Bigtable instance and table.
* Define appropriate column families.
* Load data from summary tables into Bigtable.
* Run analytical queries on Bigtable rows.

---

### Bigtable setup and authentication

[This is based on Week02's seminar example]


In [20]:
# -----------------------------------------------
# Ensure latest version of Bigtable GCP connector
# -----------------------------------------------
!pip install --upgrade google-cloud-bigtable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.3/540.3 kB 49.6 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-bigtable
    Found existing installation: google-cloud-bigtable 2.21.0
    Uninstalling google-cloud-bigtable-2.21.0:
      Successfully uninstalled google-cloud-bigtable-2.21.0


In [21]:
# -----------------------------------------------
# Client library for authentication in GCP
# -----------------------------------------------
from oauth2client.client import GoogleCredentials

In [22]:
# --------------------------------------------------------
# If Bigtable API is enabled, import necessary libraries
# --------------------------------------------------------
from google.cloud import bigtable
from google.cloud.bigtable import enums
from google.cloud.bigtable.column_family import MaxVersionsGCRule
import datetime

In [23]:
# -----------------------------------------------
# Retrieve GCP credentials
# -----------------------------------------------
credentials = GoogleCredentials.get_application_default()
print(credentials)

In [24]:
# -----------------------------------------------
# Bigtable client connection
# -----------------------------------------------
# REPLACE WITH YOUR GCP PROJECT ID AND DESIRED ZONE
PROJECT_ID = "st446-wt2026-484307"       # Replace with your GCP project ID
ZONE = "europe-west2-b"           # Replace with your preferred zone (ideally, the same of Dataproc cluster)

# Instantiate a BigTable client
# admin=True is necessary for writing operations
client = bigtable.Client(project=PROJECT_ID, admin=True)

# Check connection by listing any instances running on Bigtable
instances = client.list_instances()
# Print out all instances; empty if no instances exist
for i in instances[0]:
    print (i.name)

projects/st446-wt2026-484307/instances/covid-instance


---

### Question 10: Designing and Populating a Bigtable Serving Layer

Using the two summary tables (`severity_df` and `health_vax_df`), design and implement a **Bigtable schema to store country-level COVID-19 performance metrics**.

You must:
* Create a Bigtable instance (`covid-instance`)
* Create a table (`covid_summary`) with appropriate column families (see below)
* Insert data from both Spark SQL summary tables
* Verify that the data was correctly inserted by reading a few sample rows from your Bigtable table

You must design a **denormalised Bigtable schema** with:
* Each row representing one country, identified by the country name (`location`) as the row key
* The following column families:
    * `meta`, with structural country information (`continent` and `population`)
    * `severity`, with pandemic outcome metrics (`max_total_cases`, `max_total_deaths`, `case_fatality_ratio`)
    * `health_vax`, with health system and vaccination metrics (`avg_hosp_patients`, `avg_icu_patients`, `vax_per_1k`)
    
**Requirements & Constraints:**
* Do not create multiple tables (only one denormalized table)
* Do not store all columns in one family
* Do not use joins inside Bigtable (1). All joins must be performed in Spark before writing to Bigtable
* Remember that Bigtable stores numeric values as encoded bytes (`utf-8`)
* Each country must appear only once
* All data must only include country-level rows with complete data entries (no NULLs), consistent with previous tasks

(1) Bigtable is designed to be a high-scalable, distributed denormalised structure, based on *column families* (groups of columns frequently accessed together). Join operations are expensive (time consuming) and unpredictable at scale, so Bigtable duplicates data to guarantee fast, single-row access.

**Expected output:** samples read from Bigtable should look like:

```
Row key: Australia
severity: case_fatality_ratio -> 0.21
severity: max_total_cases -> 11861161
severity: max_total_deaths -> 25236
health_vax: avg_hosp_patients -> 1708.11
health_vax: avg_icu_patients -> 121.09
health_vax: vax_per_1k -> 2647.56
meta: continent -> Oceania
meta: population -> 26177410
```

In [25]:
import time
from pyspark.sql import functions as F
from google.api_core.exceptions import FailedPrecondition
from google.cloud.bigtable.column_family import MaxVersionsGCRule
from google.cloud.bigtable import enums

INSTANCE_ID = "covid-instance"
CLUSTER_ID  = "covid-cluster"
TABLE_ID    = "covid_summary"

# Building the Spark df (called 'final_df')
meta_df = (covid_df
    .filter((F.col("continent").isNotNull()) & (~F.col("iso_code").like("OWID_%")))
    .groupBy("location")
    .agg(
        F.first("continent", ignorenulls=True).alias("continent"),
        F.max("population").alias("population")
    )
)

final_df = (
    severity_df
    .join(health_vax_df, on="location", how="inner")
    .join(meta_df,       on="location", how="inner")
    .select(
        "location",
        "continent", "population",
        "max_total_cases", "max_total_deaths", "case_fatality_ratio",
        "avg_hosp_patients", "avg_icu_patients", "vax_per_1k"
    )
    # complete rows only (no NULLs), consistent with prior tasks
    .dropna(subset=[
        "location", "continent", "population",
        "max_total_cases", "max_total_deaths", "case_fatality_ratio",
        "avg_hosp_patients", "avg_icu_patients", "vax_per_1k"
    ])
    .filter(F.col("population") > 0)
    .dropDuplicates(["location"])  # enforce "each country once"
)

# just checking- one row per country + no NULLs
print("severity_df rows:", severity_df.count())
print("health_vax_df rows:", health_vax_df.count())
print("final_df rows:", final_df.count())
print("final_df distinct locations:", final_df.select("location").distinct().count())


# Creating the Bigtable instance ('covid-instance')

instance = client.instance(INSTANCE_ID)

if not instance.exists():
    cluster = instance.cluster(
        CLUSTER_ID,
        location_id=ZONE,
        default_storage_type=enums.StorageType.SSD,
        serve_nodes=1
    )
    instance.create(clusters=[cluster])
    print("Created instance:", INSTANCE_ID)
else:
    print("Instance already exists:", INSTANCE_ID)


# Creating ONE table (called 'covid_summary') with 3 column families
table = instance.table(TABLE_ID)

for attempt in range(1, 13):  
    try:
        if not table.exists():
            table.create(column_families={
                "meta": MaxVersionsGCRule(1),
                "severity": MaxVersionsGCRule(1),
                "health_vax": MaxVersionsGCRule(1),
            })
            print("Created table:", TABLE_ID)
        else:
            print("Table already exists:", TABLE_ID)
        break
    except FailedPrecondition:
        print(f"Cluster not ready yet (attempt {attempt}/12). Waiting 10s...")
        time.sleep(10)

print("Tables in instance:", [t.table_id for t in instance.list_tables()])

# Inserting rows via batching (more efficient)
def b(x) -> bytes:
    return str(x).encode("utf-8")

rows = []
for r in final_df.toLocalIterator():
    row_key = r["location"]  # required: row key is country name
    dr = table.direct_row(row_key)

    # meta family
    dr.set_cell("meta", "continent", b(r["continent"]))
    dr.set_cell("meta", "population", b(r["population"]))

    # severity family
    dr.set_cell("severity", "max_total_cases", b(r["max_total_cases"]))
    dr.set_cell("severity", "max_total_deaths", b(r["max_total_deaths"]))
    dr.set_cell("severity", "case_fatality_ratio", b(r["case_fatality_ratio"]))

    # health_vax family
    dr.set_cell("health_vax", "avg_hosp_patients", b(r["avg_hosp_patients"]))
    dr.set_cell("health_vax", "avg_icu_patients", b(r["avg_icu_patients"]))
    dr.set_cell("health_vax", "vax_per_1k", b(r["vax_per_1k"]))

    rows.append(dr)

table.mutate_rows(rows)
print("Rows written (batched):", len(rows))




severity_df rows: 231


health_vax_df rows: 32


final_df rows: 32


final_df distinct locations: 32
Instance already exists: covid-instance
Table already exists: covid_summary
Tables in instance: ['covid_summary']


Rows written (batched): 32


In [26]:
# Checking a few samples and making sure it matches the expected output.
def show_bt_row(row_key: str):
    row = table.read_row(row_key.encode("utf-8"))
    if row is None:
        print(f"\nRow key: {row_key} (NOT FOUND)")
        return
    print(f"\nRow key: {row_key}")
    for cf, cols in row.cells.items():
        for col, cells in cols.items():
            print(f"{cf}: {col.decode('utf-8')} -> {cells[0].value.decode('utf-8')}")
            
show_bt_row("Australia")
show_bt_row("United States")
show_bt_row("Finland")


Row key: Australia
health_vax: avg_hosp_patients -> 1370.03
health_vax: avg_icu_patients -> 72.37
health_vax: vax_per_1k -> 2662.09
meta: continent -> Oceania
meta: population -> 26177410
severity: case_fatality_ratio -> 0.21
severity: max_total_cases -> 11861161
severity: max_total_deaths -> 25236

Row key: United States
health_vax: avg_hosp_patients -> 36706.54
health_vax: avg_icu_patients -> 7703.35
health_vax: vax_per_1k -> 2000.44
meta: continent -> North America
meta: population -> 338289856
severity: case_fatality_ratio -> 1.15
severity: max_total_cases -> 103436829
severity: max_total_deaths -> 1193165

Row key: Finland
health_vax: avg_hosp_patients -> 379.49
health_vax: avg_icu_patients -> 25.86
health_vax: vax_per_1k -> 2405.29
meta: continent -> Europe
meta: population -> 5540745
severity: case_fatality_ratio -> 0.76
severity: max_total_cases -> 1499712
severity: max_total_deaths -> 11466


### Question 11: COVID-19 Normalised Impact Score

You were asked to **engineer a composite metric to rank countries** analytically.

Using the country summary Bigtable table created in Question 8, compute a **normalised impact score** for each country, defined as:

```
normalised_burden = ((avg_hosp_patients + avg_icu_patients) / population) * 100000
```

and

```
impact_score = normalised_burden − (vax_per_1k / 1000)
```

and return the top 20 countries with the lowest impact score, sorted in ascending order.

**Requirements & Constraints:**
* For each country row, extract the necessary columns from the relevant column families
* Skip rows with missing required values
* Output must contain exactly 20 rows
* Do not perform joins (Bigtable is denormalised).
* Recall that values are stored as bytes (`utf-8`) in Bigtable, so numeric conversion is necessary before performing calculations

**Expected output:** DataFrame with:
* `country` (or `location`)
* `population`
* `avg_hosp_patients`
* `avg_icu_patients`
* `normalised_burden`
* `vax_per_1k`
* `impact_score`

In [27]:
# Debug: inspect what's really in Bigtable for a known key
row = table.read_row(b"Australia")
print("Row exists:", row is not None)
if row:
    print("Row key:", row.row_key.decode("utf-8"))
    for cf, cols in row.cells.items():
        print("CF:", cf)
        for q, cells in cols.items():
            print("  ", q.decode("utf-8"), "->", cells[0].value.decode("utf-8"))
            

Row exists: True
Row key: Australia
CF: health_vax
   avg_hosp_patients -> 1370.03
   avg_icu_patients -> 72.37
   vax_per_1k -> 2662.09
CF: meta
   continent -> Oceania
   population -> 26177410
CF: severity
   case_fatality_ratio -> 0.21
   max_total_cases -> 11861161
   max_total_deaths -> 25236


In [30]:
def read_float(bt_row, family: str, qualifier: str):
    fam = bt_row.cells.get(family)
    if not fam:
        return None
    cells = fam.get(qualifier.encode("utf-8"))
    if not cells:
        return None
    try:
        return float(cells[0].value.decode("utf-8"))
    except Exception:
        return None

results = []

for row in table.read_rows():  # <-- no filters
    country = row.row_key.decode("utf-8")

    population = read_float(row, "meta", "population")
    avg_hosp   = read_float(row, "health_vax", "avg_hosp_patients")
    avg_icu    = read_float(row, "health_vax", "avg_icu_patients")
    vax_per_1k = read_float(row, "health_vax", "vax_per_1k")

    if (population is None or population <= 0 or
        avg_hosp is None or avg_icu is None or vax_per_1k is None):
        continue

    normalised_burden = ((avg_hosp + avg_icu) / population) * 100000
    impact_score = normalised_burden - (vax_per_1k / 1000)

    results.append({
        "country": country,
        "population": population,
        "avg_hosp_patients": avg_hosp,
        "avg_icu_patients": avg_icu,
        "normalised_burden": normalised_burden,
        "vax_per_1k": vax_per_1k,
        "impact_score": impact_score,
    })

impact_df = pd.DataFrame(results)

print("Valid rows found:", len(impact_df))

# Must output exactly 20 rows
impact_df = impact_df.sort_values("impact_score", ascending=True).head(20)
assert len(impact_df) == 20, "Output must contain exactly 20 rows"

impact_df = impact_df.round({
    "population": 0,
    "avg_hosp_patients": 2,
    "avg_icu_patients": 2,
    "normalised_burden": 6,
    "vax_per_1k": 2,
    "impact_score": 6
})


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

impact_df

Valid rows found: 32


,country,population,avg_hosp_patients,avg_icu_patients,normalised_burden,vax_per_1k,impact_score
19,Netherlands,17564020.0,607.95,161.70,4.381970,2263.96,2.118010
0,Australia,26177410.0,1370.03,72.37,5.510094,2662.09,2.848004
8,Denmark,5882259.0,351.60,28.03,6.453813,2543.50,3.910313
18,Malaysia,33938216.0,2026.49,203.87,6.571825,2140.87,4.430955
26,South Korea,51815808.0,3510.05,307.32,7.367192,2502.09,4.865102
10,Finland,5540745.0,379.49,25.86,7.315803,2405.29,4.910513
3,Bolivia,12224114.0,703.76,82.42,6.431386,1201.77,5.229616
12,Ireland,5023108.0,389.20,18.02,8.106933,2219.40,5.887533
28,Sweden,10549349.0,807.32,111.34,8.708215,2683.84,6.024375
15,Japan,123951696.0,12171.04,706.84,10.389434,3095.95,7.293484


---

### FOLLOW-UP QUESTIONS:

Pick one of these questions and discuss on your report:

* **FQ1.** Explain why a country may have a high vaccination per 1,000 metric but still show high average ICU or hospital occupancy.
* **FQ2.** How does average ICU load correlate (visually or conceptually) with vaccination intensity?
* **FQ3.** Which continents dominate the top CFR rankings? What structural factors (age structure, reporting quality, health capacity) might explain this?
* **FQ4.** What are limitations of the impact score calculated in Question 11?